<a href="https://colab.research.google.com/github/SFcrypt/ColabUI/blob/main/Maker/Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Iniciar Proyecto 🧽**

</details>
<img src="https://i.pinimg.com/originals/c1/bc/3d/c1bc3d6ba8b7a7c9ff037660e2e4f2c2.gif" width="150%" height="200‰" style="margin-bottom: 0;">

Automatiza la creación y configuración de carpetas en Google Drive, estableciendo una estructura organizada y personalizado para los proyectos de personajes.

<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/9.0-353535?style=for-the-badge&logo=Github&label=Versi%C3%B3n&labelColor=292929" alt="My Civitai" width="110">
  </a>
</small>
<br>
<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/7.0-353535?style=for-the-badge&logo=Pointy&logoColor=%23fafafa&label=Manager&labelColor=292929" alt="My Civitai" width="112">
  </a>
</small>

---

<details>
  <summary><font color=gray>Disclaimer</font></summary>

```diff
⭕ Descargo de responsabilidad
+ Herramientas para gestión de archivos y proyectos en Google Drive  
+ Cumplimiento de las directrices de Google Colab  
+ Adherencia a los Términos de servicio de Google Colab  
```

</details>

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Añadido soporte para etiquetado automático con WaifuDiffusion Tagger  
+ Mejorada la interfaz interactiva para filtrar y recortar imágenes  
+ Implementada función para renombrar y convertir imágenes en lote  
+ Integrada herramienta para comprimir y descomprimir archivos .zip  
+ Añadido conteo de archivos en carpetas y subcarpetas  
+ Simplificada la estructura de carpetas generadas en Google Drive  
+ Corregidos errores al montar Google Drive  
```

</details>

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Optimizaciones en la compresión y descompresión de archivos  
+ Mejora en la configuración inicial de Colab  
+ Reducción del tiempo de instalación de dependencias  
+ Actualizaciones de bibliotecas críticas como `onnxruntime`, `ultralytics`, `Pillow`, y más  
+ Añadido soporte para conteo de archivos específicos (imágenes, textos, modelos)  
+ Simplificada la integración con Google Drive  
+ Mejor manejo de errores al crear carpetas en Google Drive  
```

In [ ]:
# configurar proyecto

#@markdown ### **Configuración del proyecto**
#@markdown este bloque inicializa el proyecto en google colab, prepara el entorno
#@markdown de trabajo y define la estructura principal donde se guardarán los archivos.

# módulos necesarios
import os, re, io, time
from IPython import get_ipython
import ipywidgets as widgets
from IPython.display import clear_output, display, HTML

#@markdown también descarga archivos desde **github**
#@markdown y muestra mensajes visuales para confirmar que cada paso se completó correctamente. <p>

# conectar google drive
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# clonar colabui
if not os.path.exists("ColabUI"):
    !git clone https://github.com/SFcrypt/ColabUI.git
    clear_output()

# importar widgets visuales (global)
from ColabUI.Animated_box import (
    load_style, pink_button_download)

from ColabUI.Animated_progress import (
    Animated_progress)

# función configurar Nombre
def configurar_proyecto(b):
    clear_output(wait=True)

    global ruta_name
    nombre = nombre_input.value.strip().lower()

    # si no hay nombre salir
    if not nombre:
        return

    # definir rutas
    ruta_name  = nombre
    ruta_drive = "drive/MyDrive/"
    ruta_proy  = os.path.join(ruta_drive, "Loras")
    ruta_data  = os.path.join(ruta_proy, ruta_name, "dataset")

    os.makedirs(ruta_data, exist_ok=True)
    clear_output()
    load_style()

    # mostrar progreso con animación
    progress = Animated_progress(total=5,
    title="Creando proyecto")

    # pequeña animación
    for i in range(1, 8):
        time.sleep(0.3)
        progress.update(step=1, text=f"")

    # completar con título final
    progress.button_text = ruta_name
    progress.complete("Proyecto listo")

# Crear input centrado y botón rosa
init_btn, nombre_input = pink_button_download(
    title="Crear proyecto",
    btn_text="Crear",
    input_placeholder="Nombre del proyecto")

# vincular evento click
init_btn.on_click(configurar_proyecto)

# fin

In [ ]:
# Descargar Proyecto

#@markdown ### **Descargar proyecto**
#@markdown este script permite descargar un archivo .zip desde un link directo de google drive,
#@markdown descomprimirlo automáticamente en tu drive y eliminar el zip al finalizar.
#@markdown además, actualiza automáticamente el nombre del `proyecto`
#@markdown basado en el archivo zip, incluye barra de progreso personalizada.

# importaciones y autenticar Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, zipfile, logging, os, re
from IPython.display import clear_output

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Estilo y ruta
load_style()
ruta_drive = "drive/MyDrive/"

# Widget descarga
download_btn, link_input = pink_button_download(
    title="Descargar proyecto",
    btn_text="Descargar",
    input_placeholder="Link del proyecto")

# Extraer ID del link
def extract_file_id_from_url(url):
    match = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if not match: raise Exception("❌ Enlace incorrecto")
    return match.group(1)

# Descargar y extraer zip
def download_and_extract(file_id, nombre, ruta_destino):
    global ruta_name

    # Obtener info del archivo
    meta = drive_service.files().get(fileId=file_id, fields="size, name").execute()
    file_size = int(meta['size'])
    file_name = meta.get('name', nombre)
    output_path = os.path.join(ruta_destino, file_name)

    # Definir ruta_name como nombre del zip
    ruta_name = os.path.splitext(file_name)[0]
    progress = Animated_progress(total=file_size, title="Descargando")

    # Descargar archivo
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.FileIO(output_path, 'wb')
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        progress.update(int(status.progress() * file_size))

    # Extraer zip
    with zipfile.ZipFile(output_path, 'r') as zipf:
        for f in zipf.infolist():
            zipf.extract(f, ruta_destino)

    # Actualizar barra y eliminar zip
    progress.button_text = ruta_name
    progress.complete("Proyecto listo")
    os.remove(output_path)

# Descargar proyecto al click
def descargar_proyecto(b):
    clear_output(wait=True)
    Link = link_input.value.strip()
    if not Link: return
    try:
        file_id = extract_file_id_from_url(Link)
        download_and_extract(file_id, "", ruta_drive)
    except Exception as e:
        print(f"Error: {e}")

download_btn.on_click(descargar_proyecto)

# Fin

In [ ]:
# Comprimir Proyecto

#@markdown ### **Comprimir proyecto**
#@markdown este script permite comprimir automáticamente `dataset` de un proyecto en un archivo `zip`
#@markdown la compresión solo se realizará si la carpeta contiene más de 5 imágenes.
#@markdown es útil para hacer respaldos rápidos de tus proyectos y mantenerlos organizados.

# importaciones
import os, zipfile
from IPython.display import clear_output

# rutas dinámicas
ruta_drive  = "drive/MyDrive/"
ruta_proy   = os.path.join(ruta_drive, "Loras")
ruta_back   = os.path.join(ruta_drive, "Backup")
ruta_data   = os.path.join(ruta_proy, ruta_name, "dataset")

os.makedirs(ruta_data, exist_ok=True)
os.makedirs(ruta_back, exist_ok=True)

# función para comprimir 'dataset' solo si hay más de 5 imágenes
def comprimir_dataset(ruta_carpeta, ruta_zip):
    try:
        imagenes = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        if len(imagenes) <= 5:
            print("error: ❌ imágenes insuficientes")
            return

        # obtener todos los archivos de dataset
        archivos = [os.path.join(r, f) for r, d, fs in os.walk(ruta_carpeta) for f in fs]

        # barra de progreso
        progress = Animated_progress(
        total=len(archivos),
        title="comprimiendo")

        # crear zip
        with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in archivos:
                arcname = os.path.join("Loras", ruta_name, "dataset", os.path.relpath(f, start=ruta_carpeta))
                zipf.write(f, arcname)
                progress.update(1)

        # actualizar botón con número de imágenes
        progress.button_text = f"{len(imagenes)} imágenes"
        progress.complete("Proyecto listo")

    except Exception as e:
        print(f"error: ❌ comprimiendo: {e}")

# ejecutar compresión automáticamente
ruta_zip = os.path.join(ruta_back, f"{ruta_name}.zip")
comprimir_dataset(ruta_data, ruta_zip)

#Fin


## **Crear Lora 🦜**
[![My Github](https://img.shields.io/badge/GitHub-292929?style=for-the-badge&logo=GitHub)](https://github.com/TuUsuario)
[![My Civitai](https://img.shields.io/badge/Colab-%23292929?style=for-the-badge&logo=googlecolab)](https://civitai.com)

Herramientas integrales para etiquetar, filtrar, recortar, renombrar, comprimir, descomprimir, y descargar. Optimiza la organización y el procesamiento de tus proyectos con IA y automatización.

<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/1.0-353535?style=for-the-badge&logo=Github&label=Versi%C3%B3n&labelColor=292929" alt="My Civitai" width="110">
  </a>
</small>
<br>
<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/2.5-353535?style=for-the-badge&logo=Pointy&logoColor=%23fafafa&label=Manager&labelColor=292929" alt="My Civitai" width="112">
  </a>
</small>

---

<details>
  <summary><font color=gray>Disclaimer</font></summary>

```diff
⭕ Descargo de responsabilidad
+ Herramientas para gestión de archivos y proyectos en Google Drive  
+ Cumplimiento de las directrices de Google Colab  
+ Adherencia a los Términos de servicio de Google Colab  
```

</details>

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Añadido soporte para etiquetado automático con WaifuDiffusion Tagger  
+ Mejorada la interfaz interactiva para filtrar y recortar imágenes  
+ Implementada función para renombrar y convertir imágenes en lote  
+ Integrada herramienta para comprimir y descomprimir archivos .zip  
+ Añadido conteo de archivos en carpetas y subcarpetas  
+ Simplificada la estructura de carpetas generadas en Google Drive  
+ Corregidos errores al montar Google Drive  
```

</details>

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Optimizaciones en la compresión y descompresión de archivos  
+ Mejora en la configuración inicial de Colab  
+ Reducción del tiempo de instalación de dependencias  
+ Actualizaciones de bibliotecas críticas como `onnxruntime`, `ultralytics`, `Pillow`, y más  
+ Añadido soporte para conteo de archivos específicos (imágenes, textos, modelos)  
+ Simplificada la integración con Google Drive  
+ Mejor manejo de errores al crear carpetas en Google Drive  
```

In [ ]:
#@markdown ### **calculadora de entrenamiento**
#@markdown calcula cuántos pasos totales tendrá el entrenamiento.
#@markdown recomendado para sdxl lora: `500` `o` `700` `pasos`

import os
from pathlib import Path

# calcula el total de pasos de entrenamiento
def calculate_total_steps(train_batch_size, num_epochs, num_train_images, num_repeats):
    """Calculates the total training steps.

    Args:
        train_batch_size: Batch size for training.
        num_epochs: Number of training epochs.
        num_train_images: Total number of training images.
        repeats: Number of times to repeat the dataset.

    Returns:
        The total number of training steps.
    """
    steps_per_epoch = num_train_images * num_repeats // train_batch_size
    total_steps = steps_per_epoch * num_epochs
    return total_steps

# cuenta imágenes del dataset de forma recursiva
def count_images(directory_path):
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}
    total_count = 0
    extension_counts = {}

    try:
        path = Path(directory_path)
        if not path.exists():
            raise FileNotFoundError(f"Directory not found: {directory_path}")

        for file in path.glob('**/*'):  # búsqueda recursiva
            if file.is_file():
                ext = file.suffix.lower()
                if ext in image_extensions:
                    total_count += 1
                    extension_counts[ext] = extension_counts.get(ext, 0) + 1

        return total_count, extension_counts

    except Exception as e:
        print(f"❌ error: {str(e)}")
        return 0, {}

# directorio del dataset de entrenamiento
ruta_drive = "drive/MyDrive/"
ruta_proy  = os.path.join(ruta_drive, "Loras")
ruta_data  = os.path.join(ruta_proy, ruta_name, "dataset")

directory = ruta_data
total, by_extension = count_images(directory)

# badge: resumen del dataset
clear_output()
print("\n📋 [dataset]")
print(f"   🎴 \033[34m{total}\033[0m imágenes")

# parámetros principales de entrenamiento
Tamaño_de_entrenamiento = 5  # @param {"type":"number"}
train_batch_size = Tamaño_de_entrenamiento

Cantidad_Épocas = 5  # @param {"type":"number"}
num_epochs = Cantidad_Épocas
num_train_images = total

Repeticiones = 10       # @param {"type":"number"}
num_repeats = Repeticiones

# cálculo final de pasos totales
total_steps = calculate_total_steps(
    train_batch_size,
    num_epochs,
    num_train_images,
    num_repeats
)

# badge: resultado final
print("\n🐝 [resultado]")
print(f"   ⚙️ pasos: \033[34m{total_steps}\033[0m")

# badge: recomendación visual
if total_steps < 450:
    print("   🟡 bajo: Aumenta repeticiones")
elif total_steps > 750:
    print("   🔴 alto: sobreentrenamiento")
else:
    print("   🟢 óptimo: para lora")

#Fin

In [ ]:
# Iniciar Configurar

#@markdown ### **Procesamiento**
#@markdown Decide el modelo que se descargará y se usará para entrenamiento
#@markdown Usar un modelo diffusers consume menos recursos. Las 3 opciones funcionarán con o sin diffusers.

# Librería estándar
import os, re, json, glob, time, ast, zipfile, shutil, subprocess
from pathlib import Path
from urllib.parse import urlparse, unquote
from subprocess import getoutput

# Terceros
import requests, gdown, toml, torch
from huggingface_hub import HfFileSystem
from huggingface_hub.utils import validate_repo_id, HfHubHTTPError

# IPython Colab
from IPython.display import clear_output
from IPython.utils import capture
from google.colab import drive

%store -d train_data_dir

# Descargar Proyecto
root_dir = "/content"
ruta_drive = "/content/drive/MyDrive"
ruta_proy = os.path.join(ruta_drive, "Loras", ruta_name)
ruta_data = os.path.join(ruta_proy, "dataset")
ruta_safe = os.path.join(ruta_proy, "models")
train_data_dir = os.path.join(root_dir, "LoRA", ruta_name)
os.makedirs(train_data_dir, exist_ok=True)

# Copiar proyecto
def copy_dataset_to_local():
    local_zip = "/content/dataset.zip"

    # si ya hay archivos, no hacer nada
    for f in os.listdir(train_data_dir):
        if f.lower().endswith((".png", ".jpg", ".txt")):
            return

    # crear zip Directamente en colab
    !cd "{ruta_data}" && zip -r "{local_zip}" .
    !unzip -oq "{local_zip}" -d "{train_data_dir}"
    !rm -f "{local_zip}"

copy_dataset_to_local()
clear_output()
print("📦 dataset listo...")

# bandera de instalación
kohya_flag = "/content/ColabUI/kohya.txt"
if not os.path.exists(kohya_flag):

    # configurar rutas principales
    venv = f"{root_dir}/venv"
    py = f"{venv}/bin/python"
    pi = f"{venv}/bin/pip"

    # instalar python y venv
    !apt-get update -qq
    !apt-get install -qq -y python3.10 python3.10-venv python3.10-distutils
    !python3.10 -m venv {venv}
    !{py} -m ensurepip --upgrade
    !{pi} install -U uv

    # activar entorno virtual
    os.environ["VIRTUAL_ENV"] = venv
    os.environ["PATH"] = f"{venv}/bin:" + os.environ["PATH"]
    clear_output()

# función auxiliar para instalar paquetes con uv
def uvp(x=None, r=None):
    !uv pip install -p {py} {'-r '+r if r else x}

# recuperar variables almacenadas
%store -r

# definir estructura de directorios
drive_dir         = os.path.join(root_dir, "drive/MyDrive")
deps_dir          = os.path.join(root_dir, "deps")
repo_dir          = os.path.join(root_dir, "kohya-trainer")
training_dir      = os.path.join(root_dir, "LoRA")
pretrained_model  = os.path.join(root_dir, "pretrained_model")
vae_dir           = os.path.join(root_dir, "vae")
lora_dir          = os.path.join(root_dir, "network_weight")
repositories_dir  = os.path.join(root_dir, "repositories")
config_dir        = os.path.join(training_dir, "config")
tools_dir         = os.path.join(repo_dir, "tools")
finetune_dir      = os.path.join(repo_dir, "finetune")
accelerate_config = os.path.join(repo_dir, "accelerate_config/config.yaml")

# guardar rutas para otras celdas
for store in [
    "root_dir", "repo_dir", "training_dir", "pretrained_model",
    "vae_dir", "repositories_dir", "accelerate_config",
    "tools_dir", "finetune_dir", "config_dir"]:
    with capture.capture_output() as cap:
        %store {store}
        del cap

# repositorio fijo
repo_url = "https://github.com/qaneel/kohya-trainer"
branch   = "main"
use_tcmalloc   = True

%store train_data_dir

# crear carpeta de datos de entrenamiento
os.makedirs(train_data_dir, exist_ok=True)
print(f"directorio de entrenamiento: {train_data_dir}")

# clonar repositorio si no existe
def clone_repo(url, dir, branch):
    if not os.path.exists(dir):
        !git clone -b {branch} {url} {dir}

# crear estructura de carpetas necesaria
def setup_directories():
    global output_dir
    output_dir = os.path.join(ruta_safe)
    for dir in [
        training_dir, config_dir, pretrained_model,
        vae_dir, repositories_dir, output_dir]:
        os.makedirs(dir, exist_ok=True)

# instalar dependencias del proyecto
def install_dependencies():
    requirements_file = os.path.join(root_dir, "requirements.txt")
    gpu_info = getoutput("nvidia-smi")

    !apt install -qq aria2 lz4
    !git clone https://github.com/corkborg/wd14-tagger-standalone /content/wd14_tagger
    !wget -q https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
    !wget -q https://github.com/DEX-1101/kohya-trainer/raw/refs/heads/dev/requirements.txt -O /content/requirements.txt

    # instalar dependencias base
    uvp(r=f"{requirements_file}")

    # instalar dependencias extra
    uvp(
        "xformers==0.0.32.post2 fairscale pandas onnxruntime-gpu "
        "tensorflow==2.14.0 protobuf==3.20.3 voluptuous "
        "opencv-python-headless==4.9.0.80 python-dotenv "
        "onnxruntime==1.17.3 prodigyopt invisible-watermark==0.2.0 "
        "flax==0.8.4 jax==0.4.23 jaxlib==0.4.23 httpx==0.28.1 "
        "numpy==1.26.4 wandb==0.21.0 diffusers==0.18.2 jedi==0.19.2"
    )

    # instalar pytorch con cuda
    uvp(
        "torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
        "--index-url https://download.pytorch.org/whl/cu129"
    )

    # crear configuración de accelerate
    from accelerate.utils import write_basic_config
    if not os.path.exists(accelerate_config):
        write_basic_config(save_location=accelerate_config)

# preparar variables de entorno y optimizaciones
def prepare_environment():
    if use_tcmalloc:
        os.environ["LD_PRELOAD"] = f"{root_dir}/libtcmalloc_minimal.so.4"

    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["SAFETENSORS_FAST_GPU"] = "1"

    import warnings, logging
    warnings.filterwarnings("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"
    logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# flujo principal
def main():
    os.chdir(root_dir)
    clone_repo(repo_url, repo_dir, branch)
    os.chdir(repo_dir)
    setup_directories()
    install_dependencies()
    prepare_environment()

# ejecutar instalación
main()
clear_output()

# mostrar versiones activas
!echo -e "\033[0;34mPython:\033[0m \033[0;32m$(python --version 2>&1 | sed 's/Python //')\033[0m | \
\033[0;36muv:\033[0m \033[0;35m$(uv --version 2>&1 | sed 's/uv //')\033[0m"

# verificar dependencias
result = subprocess.run(
    ["uv", "pip", "check"],
    capture_output=True,
    text=True)

if result.stdout:
    for line in result.stdout.splitlines():
        print(f"- {line}")
else:
    print("\033[0;33minstalación completada correctamente\033[0m")

# crea archivo de bandera
with open(kohya_flag, 'w') as f: f.write("kohya")

# Selección de modelos
Modelo_Entrenamiento = "WAI Illustrious V15"  #@param ["Blender XL", "Illustrious XL", "Margarita V2", "Animagine V3", "WAI Illustrious V15"]
SDXL_MODEL_NAME = Modelo_Entrenamiento

Modelo_vae = "Original VAE"  #@param ["None", "FP16 VAE", "Original VAE"]
SDXL_VAE_NAME = Modelo_vae

HUGGINGFACE_TOKEN    = False
LOAD_DIFFUSERS_MODEL = True

# URLs de modelos
MODEL_URLS = {
    "Blender XL"          : "https://huggingface.co/ParahumanSkitter/Blender-XL-Illustrious-V10/resolve/main/BlenderXL.safetensors",
    "Illustrious XL"      : "https://huggingface.co/Ine007/waiIllustriousSDXL_v150/resolve/main/Illustrious_v150.safetensors",
    "Margarita V2"        : "https://huggingface.co/gsdf/CounterfeitXL/resolve/main/CounterfeitXL_%CE%B2.safetensors",
    "Animagine V3"        : "https://huggingface.co/cagliostrolab/animagine-xl-3.0/resolve/main/animagine-xl-3.0.safetensors",
    "WAI Illustrious V15" : "https://huggingface.co/Ine007/waiIllustriousSDXL_v150/resolve/main/waiIllustriousSDXL_v150.safetensors",}

VAE_URLS = {
    "None"         : "",
    "Original VAE" : "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors",
    "FP16 VAE"     : "https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors",}

SDXL_MODEL_URL = MODEL_URLS.get(SDXL_MODEL_NAME, SDXL_MODEL_NAME)
SDXL_VAE_URL   = VAE_URLS.get(SDXL_VAE_NAME, SDXL_VAE_NAME)

# Funciones
def get_filename(url):
    if any(url.endswith(ext) for ext in [".ckpt", ".safetensors", ".pt", ".pth"]):
        return os.path.basename(url)

    response = requests.get(url, stream=True)
    response.raise_for_status()

    if "content-disposition" in response.headers:
        return re.findall('filename="?([^"]+)"?', response.headers["content-disposition"])[0]
    return unquote(os.path.basename(urlparse(url).path))

def aria2_download(dst, filename, url):
    user_header = f"Authorization: Bearer {HUGGINGFACE_TOKEN}"
    subprocess.run([
        "aria2c",
        "--console-log-level=error",
        "--summary-interval=10",
        "--continue=true",
        "--max-connection-per-server=16",
        "--min-split-size=1M",
        "--split=16",
        f"--dir={dst}",
        f"--out={filename}",
        f"--header={user_header}" if "huggingface.co" in url else "",
        url])

def download(url, dst):
    filename = get_filename(url)
    filepath = os.path.join(dst, filename)
    if "drive.google.com" in url:
        gdown.download(url, filepath, quiet=False)
    else:
        if "huggingface.co" in url and "/blob/" in url:
            url = url.replace("/blob/", "/resolve/")
        aria2_download(dst, filename, url)
    return filepath

def all_folders_present(base_model_url, folders):
    fs = HfFileSystem()
    existing = set(fs.ls(base_model_url, detail=False))
    return all(f"{base_model_url}/{f}" in existing for f in folders)

def get_total_ram_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if "MemTotal" in line:
                return int(line.split()[1]) / (1024**2)

def get_gpu_name():
    try:
        return subprocess.check_output(
            "nvidia-smi --query-gpu=name --format=csv,noheader,nounits",
            shell=True
        ).decode().strip()
    except:
        return None

def main():
    global model_path, vae_path, LOAD_DIFFUSERS_MODEL
    model_path = vae_path = None
    required_folders = [
        "scheduler", "text_encoder", "text_encoder_2",
        "tokenizer", "tokenizer_2", "unet", "vae"]

    targets = {
        "model": (SDXL_MODEL_URL, "/content"),
        "vae"  : (SDXL_VAE_URL, "/content"),}

    if get_total_ram_gb() < 13 and get_gpu_name() in ["Tesla T4", "Tesla V100"]:
        LOAD_DIFFUSERS_MODEL = True
    for target, (url, dst) in targets.items():
        if not url:
            continue

        if target == "model" and LOAD_DIFFUSERS_MODEL and "huggingface.co" in url:
            match = re.search(r"huggingface\.co/([^/]+)/([^/]+)", url)
            if match:
                repo = f"{match.group(1)}/{match.group(2)}"
                if all_folders_present(repo, required_folders):
                    model_path = repo
                    continue

        path = download(url, dst)
        if target == "model":
            model_path = path
        else:
            vae_path = path

    if model_path:
        print(f"Selected model: \033[34m{model_path}\033[0m")
    if vae_path:
        print(f"Selected VAE: \033[34m{vae_path}\033[0m")

main()

#@markdown #### **Etiquetas**
#@markdown Mezclar etiquetas de anime en su lugar mejora el aprendizaje y el prompting. Una etiqueta de activación va al inicio de cada archivo de texto y no se mezclará.
Activación  = "1"  #@param [0,1,2,3]
keep_tokens = int(Activación)

#@markdown #### **Pasos**
#@markdown Tus imágenes se repetirán durante el entrenamiento. Recomiendo que sean alrededor de `50` `imágenes`.
Repeticiones = 10  #@param {type:"number"}
num_repeats  = Repeticiones
resolution   = 1024

#@markdown #### **Entrenamiento**
#@markdown Elige cuánto tiempo quieres entrenar. Un buen punto de partida es alrededor de 10 épocas o 500 pasos,
#@markdown Una época es un número de pasos igual a: tu número de imágenes multiplicado por sus repeticiones, y la cantidad de `.safetensors` a crear.<p>
Cantidad_Épocas = 5  #@param {type:"slider", min:1, max:10, step:1}
num_epochs = Cantidad_Épocas

# rutas de salida para metadatos y buckets
bucketing_json    = os.path.join(training_dir, "meta_lat.json")
metadata_json     = os.path.join(training_dir, "meta_clean.json")

# resolución base para los buckets
bucket_resolution = 1024  # resolución base para los buckets
mixed_precision   = "no"  # precisión mixta usada durante
skip_existing     = False # omitir archivos que ya existen
flip_aug          = False # aplicar aumento por volteo horizontal
clean_caption     = False # limpiar etiquetas
recursive         = True  # procesar subcarpetas

# configuración para generación de metadatos
metadata_config = {
    "_train_data_dir": train_data_dir,
    "_out_json": metadata_json,
    "recursive": recursive,
    "full_path": recursive,
    "clean_caption": clean_caption}

# configuración para bucketing y generación de latentes
bucketing_config = {
    "_train_data_dir": train_data_dir,
    "_in_json": metadata_json,
    "_out_json": bucketing_json,
    "_model_name_or_path": vae_path if vae_path else model_path,
    "recursive": recursive,
    "full_path": recursive,
    "flip_aug": flip_aug,
    "skip_existing": skip_existing,
    "batch_size": 4,
    "max_data_loader_n_workers": 2,
    "max_resolution": f"{bucket_resolution}, {bucket_resolution}",
    "mixed_precision": mixed_precision,}

# construir argumentos de línea de comandos desde un diccionario
def generate_args(config):
    args = []
    for k, v in config.items():
        if k.startswith("_"):
            args.append(f'"{v}"')
        elif isinstance(v, str):
            args.append(f'--{k}="{v}"')
        elif isinstance(v, bool) and v:
            args.append(f"--{k}")
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            args.append(f"--{k}={v}")
    return " ".join(args)

# generar comandos
merge_metadata_args = generate_args(metadata_config)
prepare_buckets_args = generate_args(bucketing_config)

merge_metadata_command = f"python merge_all_to_metadata.py {merge_metadata_args}"
prepare_buckets_command = f"python prepare_buckets_latents.py {prepare_buckets_args}"

# verificar si ya existen archivos .npz (latentes)
def npz_exists(data_dir):
    return len(glob.glob(os.path.join(data_dir, "**/*.npz"), recursive=True)) > 0

latents_exist = npz_exists(train_data_dir)

# ejecutar scripts de kohya
os.chdir(finetune_dir)
os.environ["PYTHONPATH"] = "/content/kohya-trainer"
!{merge_metadata_command}
time.sleep(1)
if latents_exist:
    print("Latentes detectados")
else:
    print("generando latentes...")
    !{prepare_buckets_command}
clear_output()

#@markdown ####  **Aprendizaje**
#@markdown El scheduler es el algoritmo que guía la tasa de aprendizaje. Si no estás seguro, elige `constant` y ignora el número. Recomiendo `cosine_with_restarts` con 3 reinicios.
lr_scheduler     = "cosine_with_restarts"  # @param ["linear", "cosine", "cosine_with_restarts", "polynomial", "constant", "constant_with_warmup", "adafactor"]
lr_scheduler_num = "3"  # @param ["1", "2", "3", "4", "5"]
lr_warmup_steps  = 100 # pasos de calentamiento

#@markdown ####  **Estructura**
#@markdown LoRA es el tipo clásico y bueno para una variedad de propósitos. LoCon es bueno con estilos artísticos ya que tiene más capas para aprender más aspectos del dataset.
Tipo_de_lora     = "LoRA_LierLa" # @param ["LoRA_LierLa", "LoRA_C3Lier", "DyLoRA_LierLa", "DyLoRA_C3Lier", "LoCon", "LoHa", "IA3", "LoKR", "DyLoRA_Lycoris"]
network_category = Tipo_de_lora
network_args     = ""  # argumentos adicionales de red

#@markdown Más dim significa LoRA más grande, puede almacenar más información pero más no siempre es mejor.
# Slider combinado para network_dim y network_alpha (network_alpha = 2 * network_dim)
Tamaño_de_lora = 8  #@param {type:"slider", min:2, max:16, step:2}
conv_dim   = Tamaño_de_lora
conv_alpha = conv_dim * 2

network_dim   = conv_alpha
network_alpha = conv_dim
unit          = 4

# procesar argumentos de red
if isinstance(network_args, str):
    network_args = network_args.strip()
    if network_args.startswith('[') and network_args.endswith(']'):
        try:
            network_args = ast.literal_eval(network_args)
        except (SyntaxError, ValueError) as e:
            print(f"Error parsing network_args: {e}\n")
            network_args = []
    elif len(network_args) > 0:
        print(f"WARNING! '{network_args}' is not a valid list! Put args like this: [\"args=1\", \"args=2\"]\n")
        network_args = []
    else:
        network_args = []

# definición de módulos según el tipo de red
network_config = {
    "LoRA_LierLa":    {"module": "networks.lora",   "args": []},
    "LoRA_C3Lier":    {"module": "networks.lora",   "args": [f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]},
    "DyLoRA_LierLa":  {"module": "networks.dylora", "args": [f"unit={unit}"]},
    "DyLoRA_C3Lier":  {"module": "networks.dylora", "args": [f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}", f"unit={unit}"]},
    "LoCon":          {"module": "lycoris.kohya",   "args": ["algo=locon",  f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]},
    "LoHa":           {"module": "lycoris.kohya",   "args": ["algo=loha",   f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]},
    "IA3":            {"module": "lycoris.kohya",   "args": ["algo=ia3",    f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]},
    "LoKR":           {"module": "lycoris.kohya",   "args": ["algo=lokr",   f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]},
    "DyLoRA_Lycoris": {"module": "lycoris.kohya",   "args": ["algo=dylora", f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]},}

# seleccionar módulo y argumentos finales
network_module = network_config[network_category]["module"]
network_args.extend(network_config[network_category]["args"])

# configuración final de lora
lora_config = {
    "additional_network_arguments": {
    "no_metadata"      : False,
    "network_module"   : network_module,
    "network_dim"      : network_dim,
    "network_alpha"    : network_alpha,
    "network_args"     : network_args,
    "training_comment" : None, },}

print(toml.dumps(lora_config))

# titulo configuración del optimizador
# @title ## **Optimizer Config**

learning_rate   = 5e-4   # tasa de aprendizaje base
text_encoder_lr = None # tasa de aprendizaje del encoder

#@markdown ####  **Entrenamiento**
#@markdown Ajusta estos parámetros según la configuración de tu Colab.
#@markdown Si estás usando la versión gratuita, deberías seleccionar un modelo diffusers al inicio de esta celda.
#@markdown Un batch size más alto suele ser más rápido pero usa más memoria.
Tamaño_de_entrenamiento = 5  #@param {type:"number"}
train_batch_size = Tamaño_de_entrenamiento

#@markdown ####  **Avanzado**
#@markdown El optimizador es el algoritmo usado para entrenar. AdamW8Bit es el predeterminado y funciona muy bien, mientras que Prodigy
#@markdown ajusta automáticamente la tasa de aprendizaje y puede tener varias ventajas como entrenar más rápido.
optimizer_type = "AdaFactor" # @param ["AdamW", "AdamW8bit", "Lion8bit", "Lion", "SGDNesterov", "SGDNesterov8bit", "DAdaptation(DAdaptAdamPreprint)", "DAdaptAdaGrad", "DAdaptAdam", "DAdaptAdan", "DAdaptAdanIP", "DAdaptLion", "DAdaptSGD", "AdaFactor", "Prodigy"]
optimizer_args = "[\"scale_parameter=False\", \"relative_step=False\", \"warmup_init=False\" ]"

# procesar argumentos del optimizador
if isinstance(optimizer_args, str):
    optimizer_args = optimizer_args.strip()
    if optimizer_args.startswith('[') and optimizer_args.endswith(']'):
        try:
            optimizer_args = ast.literal_eval(optimizer_args)
        except (SyntaxError, ValueError) as e:
            print(f"Error parsing optimizer_args: {e}\n")
            optimizer_args = []
    elif len(optimizer_args) > 0:
        print(f"WARNING! '{optimizer_args}' is not a valid list! Put args like this: [\"args=1\", \"args=2\"]\n")
        optimizer_args = []
    else:
        optimizer_args = []

# configuración final del optimizador
optimizer_config = {
    "optimizer_arguments": {
    "optimizer_type"          : optimizer_type,
    "learning_rate"           : learning_rate,
    "text_encoder_lr"         : text_encoder_lr,
    "network_train_unet_only" : False if text_encoder_lr else True,
    "max_grad_norm"           : 0,
    "optimizer_args"          : optimizer_args,
    "lr_scheduler"            : lr_scheduler,
    "lr_warmup_steps"         : lr_warmup_steps,
    "lr_scheduler_num_cycles" : lr_scheduler_num if lr_scheduler == "cosine_with_restarts" else None,
    "lr_scheduler_power"      : lr_scheduler_num if lr_scheduler == "polynomial" else None,
    "lr_scheduler_type"       : None,
    "lr_scheduler_args"       : None,},}
print(toml.dumps(optimizer_config))

project_name               = "epoch"   # Nombre del proyecto
wandb_api_key              = ""        # API Key de Weights & Biases
in_json                    = "/content/LoRA/meta_lat.json"

gradient_checkpointing     = True      # Reduce VRAM a cambio de más tiempo
no_half_vae                = True      # Usa VAE en precisión completa
cache_latents              = True      # Cachea latentes en memoria
cache_latents_to_disk      = True      # Cachea latentes en disco
cache_text_encoder_outputs = False     # Cachea salidas del text

min_timestep               = 0         # Timestep mínimo de entrenamiento
max_timestep               = 1000      # Timestep máximo de entrenamiento

mixed_precision             = "fp16"     # Precisión mixta (fp16/bf16)
seed                        = -1         # semillas
optimization                = "xformers" # modelo de optimizador
save_precision              = "fp16"     # Precisión mixta
save_every_n_epochs         = 1          # Guardar cada N épocas

#@markdown Define el método de muestreo, la base de prompts automáticos según el estilo, y un prompt personalizado.
enable_sample               = False
sampler                     = "euler_a"
positive_prompt             = ""
negative_prompt             = ""
quality_prompt              = "Waifu Diffusion 1.5"
if quality_prompt          == "NovelAI":
    positive_prompt         = "masterpiece, best quality, "
    negative_prompt         = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry, "
if quality_prompt          == "AbyssOrangeMix":
    positive_prompt         = "masterpiece, best quality, "
    negative_prompt         = "(worst quality, low quality:1.4), "
if quality_prompt          == "Stable Diffusion XL":
    negative_prompt         = "3d render, smooth, plastic, blurry, grainy, low-resolution, deep-fried, oversaturated"
custom_prompt               = ""

prompt_from_caption         = "none"
if prompt_from_caption     != "none":
    custom_prompt           = ""
num_prompt                  = 2
logging_dir                 = os.path.join(training_dir, "logs")
lowram                      = int(next(line.split()[1] for line in open('/proc/meminfo') if "MemTotal" in line)) / (1024**2) < 15

os.chdir(repo_dir)

prompt_config = {
    "prompt": {
        "negative_prompt" : negative_prompt,
        "width"           : resolution,
        "height"          : resolution,
        "scale"           : 12,
        "sample_steps"    : 28,
        "subset"          : [],
    }}

train_config = {
    "sdxl_arguments": {
        "cache_text_encoder_outputs"    : cache_text_encoder_outputs,
        "no_half_vae"                   : True,
        "min_timestep"                  : min_timestep,
        "max_timestep"                  : max_timestep,
        "shuffle_caption"               : True if not cache_text_encoder_outputs else False,
        "lowram"                        : lowram
    },
    "model_arguments": {
        "pretrained_model_name_or_path" : model_path,
        "vae"                           : vae_path,
    },
    "dataset_arguments": {
        "debug_dataset"                 : False,
        "in_json"                       : in_json,
        "train_data_dir"                : train_data_dir,
        "dataset_repeats"               : num_repeats,
        "keep_tokens"                   : keep_tokens,
        "resolution"                    : str(resolution) + ',' + str(resolution),
        "color_aug"                     : False,
        "face_crop_aug_range"           : None,
        "token_warmup_min"              : 1,
        "token_warmup_step"             : 0,
    },
    "training_arguments": {
        "output_dir"                    : os.path.join(output_dir),
        "output_name"                   : project_name if project_name else "last",
        "save_precision"                : save_precision,
        "save_every_n_epochs"           : save_every_n_epochs,
        "save_n_epoch_ratio"            : None,
        "save_last_n_epochs"            : None,
        "resume"                        : None,
        "train_batch_size"              : train_batch_size,
        "max_token_length"              : 225,
        "mem_eff_attn"                  : False,
        "sdpa"                          : True if optimization == "scaled dot-product attention" else False,
        "xformers"                      : True if optimization == "xformers" else False,
        "max_train_epochs"              : num_epochs,
        "max_data_loader_n_workers"     : 8,
        "persistent_data_loader_workers": True,
        "seed"                          : seed if seed > 0 else None,
        "gradient_checkpointing"        : gradient_checkpointing,
        "gradient_accumulation_steps"   : 1,
        "mixed_precision"               : mixed_precision,
        "cache_latents"                 : cache_latents,
        "cache_latents_to_disk"         : cache_latents_to_disk,
    },
    "logging_arguments": {
        "log_with"                : "wandb" if wandb_api_key else "tensorboard",
        "log_tracker_name"        : project_name if wandb_api_key and not project_name == "last" else None,
        "logging_dir"             : logging_dir,
        "log_prefix"              : project_name if not wandb_api_key else None,
    },
    "sample_prompt_arguments": {
        "sample_every_n_steps"    : None,
        "sample_every_n_epochs"   : save_every_n_epochs if enable_sample else None,
        "sample_sampler"          : sampler,
    },
    "saving_arguments": {
        "save_model_as": "safetensors"},
}

def write_file(filename, contents):
    with open(filename, "w") as f:
        f.write(contents)

def prompt_convert(enable_sample, num_prompt, train_data_dir, prompt_config, custom_prompt):
    if enable_sample:
        search_pattern = os.path.join(train_data_dir, '**/*' + prompt_from_caption)
        caption_files = glob.glob(search_pattern, recursive=True)

        if not caption_files:
            if not custom_prompt:
                custom_prompt = "masterpiece, best quality, 1girl, aqua eyes, baseball cap, blonde hair, closed mouth, earrings, green background, hat, hoop earrings, jewelry, looking at viewer, shirt, short hair, simple background, solo, upper body, yellow shirt"
            new_prompt_config = prompt_config.copy()
            new_prompt_config['prompt']['subset'] = [
                {"prompt": positive_prompt + custom_prompt if positive_prompt else custom_prompt}
            ]
        else:
            selected_files = random.sample(caption_files, min(num_prompt, len(caption_files)))

            prompts = []
            for file in selected_files:
                with open(file, 'r') as f:
                    prompts.append(f.read().strip())

            new_prompt_config = prompt_config.copy()
            new_prompt_config['prompt']['subset'] = []

            for prompt in prompts:
                new_prompt = {
                    "prompt": positive_prompt + prompt if positive_prompt else prompt,
                }
                new_prompt_config['prompt']['subset'].append(new_prompt)

        return new_prompt_config
    else:
        return prompt_config

def eliminate_none_variable(config):
    for key in config:
        if isinstance(config[key], dict):
            for sub_key in config[key]:
                if config[key][sub_key] == "":
                    config[key][sub_key] = None
        elif config[key] == "":
            config[key] = None

    return config

try:
    train_config.update(optimizer_config)
except NameError:
    raise NameError("'optimizer_config' dictionary is missing. Please run  '4.1. Optimizer Config' cell.")

try:
    train_config.update(lora_config)
except NameError:
    raise NameError("'lora_config' dictionary is missing. Please run  '4.1. LoRa: Low-Rank Adaptation Config' cell.")

advanced_training_warning = False
try:
    train_config.update(advanced_training_config)
except NameError:
    advanced_training_warning = True
    pass

prompt_config = prompt_convert(enable_sample, num_prompt, train_data_dir, prompt_config, custom_prompt)

config_path         = os.path.join(config_dir, "config_file.toml")
prompt_path         = os.path.join(config_dir, "sample_prompt.toml")

config_str          = toml.dumps(eliminate_none_variable(train_config))
prompt_str          = toml.dumps(eliminate_none_variable(prompt_config))

write_file(config_path, config_str)
write_file(prompt_path, prompt_str)

%store -r
print(config_str)
os.chdir(root_dir)

if advanced_training_warning:
    import textwrap
    error_message = (
    "ADVERTENCIA: Este no es un mensaje de error, pero falta el "
    "diccionario [advanced_training_config]. Ejecuta la celda "
    "'4.2. Advanced Training Config' si tienes la intención de usarlo, "
    "o continúa con el siguiente paso.")

    wrapped_message = textwrap.fill(error_message, width=80)
    print('\033[38;2;204;102;102m' + wrapped_message + '\033[0m\n')
    pass

print(prompt_str)

# Fin

In [ ]:
# Iniciar Entorno

#@markdown ### **Iniciar entrenamiento**
#@markdown este bloque prepara la configuración del entrenamiento,
#@markdown construye los argumentos dinámicamente y ejecuta el entrenamiento
#@markdown usando `accelerate`. Al finalizar, renombra los modelos generados
#@markdown según el nombre del proyecto y el número de época.

# importaciones necesarias
import os, re
import toml
from IPython.display import clear_output

# revisa estas rutas si deseas editar los parámetros del entrenamiento

sample_prompt = "/content/LoRA/config/sample_prompt.toml"
config_file   = "/content/LoRA/config/config_file.toml"

# Leer archivo de texto
def read_file(filename):
    with open(filename, "r") as f:
        return f.read()

# Construir argumentos CLI desde diccionario
def train(config):
    args = ""
    for k, v in config.items():
        if k.startswith("_"):
            args += f'"{v}" '
        elif isinstance(v, str):
            args += f'--{k}="{v}" '
        elif isinstance(v, bool) and v:
            args += f"--{k} "
        elif isinstance(v, float):
            args += f"--{k}={v} "
        elif isinstance(v, int):
            args += f"--{k}={v} "
    return args

# Limpiar salida antes de iniciar
clear_output()

# Configuración de Accelerate
accelerate_conf = {
    "config_file": "/content/kohya-trainer/accelerate_config/config.yaml",
    "num_cpu_threads_per_process": 1,}

# Configuración del entrenamiento
train_conf = {
    "sample_prompts": sample_prompt if os.path.exists(sample_prompt) else None,
    "config_file": config_file,
    "wandb_api_key": wandb_api_key if wandb_api_key else None,}

# Construir comandos finales
accelerate_args = train(accelerate_conf)
train_args = train(train_conf)
final_args = f"accelerate launch {accelerate_args} sdxl_train_network.py {train_args}"

# Ejecutar entrenamiento
os.chdir(repo_dir)
!{final_args}
clear_output()

models_dir = os.path.join(ruta_safe)
base_name  = ruta_name

# Renombrar Archivos
files = sorted(
    [f for f in os.listdir(models_dir) if f.endswith(".safetensors")],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 9999)

for i, filename in enumerate(files, start=1):
    new_name = f"{base_name} {i:02d}.safetensors"
    old_path = os.path.join(models_dir, filename)
    new_path = os.path.join(models_dir, new_name)

    if not os.path.exists(new_path):
        os.rename(old_path, new_path)
        total_epochs = len(files)
        print(f"\nepoch {i}/{total_epochs}")
        print(new_name)


# Fin